# Data manipulation

This tutorial shows how to combine and transform `xsnow` datasets using `xarray` operations:

1. Concatenating
2. Merging
3. Broadcasting & alignment
4. Assigning variables and coords

Typical application scenarios:

- **Combining simulations** from separate locations/times (along one or multiple dimensions)
- **Merging datasets** to update entries (e.g., overwrite past forecast fields with current nowcast), applying user-selected merge options
- **Metadata** handling via `xarray`’s merge routines; you can edit attributes post-hoc

```{note}
This page focuses on dataset-level operations. For selecting subsets, see the separate tutorial on *Subsetting & filtering*.
```

## Imports & sample data
We’ll use small built-in sample datasets to keep examples fast and reproducible.

In [ ]:
import numpy as np
import xarray as xr
import xsnow
from xsnow import xsnowDataset

# Two tiny demo datasets
ts_A = xsnow.single_profile_timeseries()   # location A
ts_B = xsnow.single_profile_timeseries()   # location B (we'll rename its location ID)

# Re-label location in ts_B to simulate a distinct site
if 'location' in ts_B.dims:
    # Turn location into a string coordinate (recommended for readability)
    loc_coord = xr.DataArray(['SITE_B'], dims=('location',))
    ts_B = ts_B.assign_coords(location=loc_coord)

# Similarly, rename A's location for clarity
if 'location' in ts_A.dims:
    ts_A = ts_A.assign_coords(location=xr.DataArray(['SITE_A'], dims=('location',)))

ts_A, ts_B

## 1) Concatenating
Use `xr.concat([...], dim=...)` to **stack datasets along a dimension**. Common cases:

- **Locations**: combine independent sites into a larger network
- **Time**: extend time series
- **Realizations**: add more ensemble members or simulation variants

### 1.1 Concatenate by `location` (combine regions/sites)
Assumes the datasets share the same other coordinates (e.g., `time`, schema).

In [ ]:
# Concatenate A and B along the location dimension
ds_loc = xr.concat([ts_A.to_xarray(), ts_B.to_xarray()], dim="location")
xs_loc = xsnowDataset(ds_loc)
print(xs_loc.sizes)
list(xs_loc.coords)

### 1.2 Concatenate by `time` (extend time coverage)
Here we simulate an extension by duplicating and shifting a small time subset. In practice, you’d join different files covering adjacent periods.

In [ ]:
# Take a short slice and shift its timestamps forward to simulate a later chunk
base = ts_A.to_xarray()
chunk1 = base.isel(time=slice(0, 12))
chunk2 = chunk1.copy(deep=True)
chunk2 = chunk2.assign_coords(time=chunk1['time'] + np.timedelta64(12, 'h'))

# Concatenate along time (ensure new times don't overlap or handle duplicates explicitly)
ds_time = xr.concat([chunk1, chunk2], dim="time")
ds_time = ds_time.sortby('time')  # good hygiene
xs_time = xsnowDataset(ds_time)
xs_time['HS'].to_series().head(6)

### 1.3 Concatenate by `realization` (add ensemble members)
You can add another **realization** by introducing a new index along that dimension. If your original dataset has only one realization, stack another copy with `realization=[0, 1]` (or string names).

In [ ]:
base = ts_A.to_xarray()

# Ensure the realization dimension exists
if 'realization' not in base.dims:
    base = base.expand_dims(realization=[0])

# Create a second realization by applying a tiny perturbation
perturbed = base.copy(deep=True)
if 'HS' in perturbed:
    perturbed['HS'] = perturbed['HS'] * 1.05  # +5% variant

# Label the two realizations distinctly
base = base.assign_coords(realization=[0])
perturbed = perturbed.assign_coords(realization=[1])

ds_reals = xr.concat([base, perturbed], dim='realization')
xs_reals = xsnowDataset(ds_reals)
xs_reals['HS'].isel(realization=0).to_series().head(3), xs_reals['HS'].isel(realization=1).to_series().head(3)

```{tip}
If labels are strings, use `assign_coords(realization=['r00','r01'])` for readability.
```

## 2) Merging
Use `xr.merge([...])` to **combine variables** from different datasets, aligning on shared coordinates.

Common patterns:
- **Add complementary variables** (e.g., merge meteorological input `.smet` into a profile dataset)
- **Overwrite/conflict resolution** using `compat`, `join`, or follow-up `.update()` / `.combine_first()`

### 2.1 Basic merge: add a derived variable from another dataset
We’ll create a trivial variable in a copy and merge it back.

In [ ]:
left = ts_A.to_xarray()
right = ts_A.to_xarray().copy(deep=True)

# Create a new scalar variable (per time) in `right`
if 'HS' in right:
    right['HS_smooth'] = right['HS'].rolling(time=6, min_periods=1).mean()

ds_merged = xr.merge([left, right], compat='no_conflicts')  # fail if incompatible
xs_merged = xsnowDataset(ds_merged)
list(xs_merged.data_vars)[:6]

### 2.2 Overwrite on conflicts (e.g., nowcast replaces forecast)
You can control conflict resolution with `compat`/`join`, or simply call `.update()` after aligning coordinates.

- `compat='override'` in `xr.merge()` lets the right-hand dataset values take precedence
- `.update()` replaces overlapping values where coordinates align
- `.combine_first()` uses the left dataset and fills gaps from the right

Below, we overwrite `HS` where timestamps overlap, simulating a nowcast correction:

In [ ]:
base = ts_A.to_xarray()
newer = ts_A.to_xarray().copy(deep=True)
if 'HS' in newer:
    newer = newer.isel(time=slice(6, None))        # a later chunk
    newer['HS'] = newer['HS'] * 1.02               # small correction

# Align then update: base <- newer
base_aligned, newer_aligned = xr.align(base, newer, join='outer')
base_updated = base_aligned.update(newer_aligned)
xs_updated = xsnowDataset(base_updated)
xs_updated['HS'].to_series().head(6)

```{tip}
Cheat sheet:
- `xr.merge([A,B], compat='override', join='inner')` → prefer B’s values on conflicts, keep only intersecting coords
- `A.update(B)` → overwrite A with B where aligned
- `A.combine_first(B)` → keep A, use B as fallback for missing values
```

## 3) Broadcasting & alignment
`xarray` automatically **aligns by coordinate labels** and **broadcasts** arrays to compatible shapes. This is powerful—but you need to be aware of what aligns with what.

### 3.1 Implicit broadcasting example
Add a per-time scalar to a per-layer variable; xarray will broadcast the scalar across the layer dimension automatically when dimensions align by name.

In [ ]:
ds = ts_A.to_xarray()

if 'HS' in ds and 'z' in ds:
    per_time_scalar = (ds['HS'] * 0 + 1).rename('scale')  # shape: (location,time,slope,realization)
    # Broadcast onto a layer variable (e.g., density-like demo using z)
    scaled_z = ds['z'] * per_time_scalar
    scaled_z.name = 'z_scaled'
    ds_bcast = xr.merge([ds, scaled_z])
    xs_bcast = xsnowDataset(ds_bcast)
    xs_bcast['z_scaled'].isel(time=0).values.shape  # check broadcasted shape

```{warning} Alignment gotcha
Operations align on **coordinate labels**, not array order. If labels do not match (e.g., differing time stamps), values are paired by label and **missing pairs become NaN**. Use `xr.align(A,B, join=...)` to control alignment behavior explicitly.
```

## 4) Assigning variables and coords
Use `.assign()` and `.assign_coords()` to add or modify variables/coords in a tidy way. This is great for adding derived variables, flags, or human-readable coordinate labels.

In [ ]:
ds = ts_A.to_xarray()

# Example: add a boolean mask (snow on ground), attach as data
has_snow = (ds['HS'].notnull()) & (ds['HS'] > 0)
ds2 = ds.assign(has_snow=has_snow)

# Example: add a readable location_name coord (string)
ds3 = ds2.assign_coords(location_name=xr.DataArray(['Station A'], dims=('location',)))

# Keep provenance
hist = ds3.attrs.get('history', '')
ds3.attrs['history'] = (hist + ('\n' if hist else '') + 'added has_snow and location_name')
xs3 = xsnowDataset(ds3)
xs3['has_snow'].isel(time=slice(0,3)).to_series().head(5)

### Summary
- **Concatenate** along a dimension to stack datasets (`location`, `time`, or `realization`)
- **Merge** to combine variables and control conflicts (`merge`, `update`, `combine_first`)
- **Broadcasting** and **alignment** are label-aware; use `xr.align` when in doubt
- **Assign** new variables and coords to keep results tidy and self-describing

Examples from the development plan:
```python
# Concatenate two regions
ds_combined = xr.concat([ds_region_A, ds_region_B], dim="location")

# Merge two simulations, letting ds_new override ds_old on conflicts
ds_merged = xr.merge([ds_old, ds_new], compat="override", join="inner")

# Many more options using merge(), .update(), or .combine_first()!
```